
# KOSIS 월간 경보 파이프라인 (범용 템플릿)

**설정 하나만 바꾸면** 새 주제에 같은 절차(검증 방식 7단계 + 통일 점진 규칙)를 적용합니다.

| 단계 | 함수 | 하는 일 |
|---|---|---|
| 1 | `fetch` | KOSIS에서 받기. 표 구조로 한 번에 받을 기간을 계산해 40,000셀 한도를 피하고, 빈 구간은 건너뜀. 저장소에 올린 파일이 있으면 그것을 씀 |
| 2 | `units` | 지역 × 연도 단위로 예측·증거·분산·규모 계산 (흐름 / 순흐름 / 재고) |
| 3 | `calibrate` | 보정 기간에서 판정 기준(α, m)을 사전 등록된 절차로 고름 |
| 4 | `backtest` | 채점 기간 성적 (전체·규모별) |
| 5 | `live` | 올해 공표분 기준 실시간 경보 목록 |
| 6 | `report` | 결과를 마크다운 한 장으로 |

**쓰는 법**
1. 아래 `TOPIC`에서 주제를 고르고 **런타임 → 모두 실행**을 누릅니다. 이미 검증한 주제(혼인, 청년 순이동, 미분양)는 키 없이 돕니다.
2. 새 주제는 맨 아래 "새 주제 추가하기"를 따라 합니다. 이때는 코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `KOSIS_API_KEY`가 필요합니다.

판정 규칙은 현재 채택한 **통일 규칙 2차(`v2`)**가 기본값입니다. 미분양 사전 검증 결과에 따라 `v5`로 바꿀 수 있습니다.


In [ ]:

import os, urllib.request, importlib
if not os.path.exists("kosis_monitor.py"):   # 코랩: 저장소에서 파이프라인 코드 받기
    urllib.request.urlretrieve("https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/kosis_monitor.py", "kosis_monitor.py")
import kosis_monitor as K; importlib.reload(K)
import pandas as pd
from IPython.display import Markdown, display
print("검증된 주제:", list(K.TOPICS))


In [ ]:

TOPIC = "청년 순이동"     # "혼인", "청년 순이동", "미분양" 중 하나 (또는 아래에서 만든 새 Topic 이름)
RULE = "v2"               # 통일 규칙 2차(현재 채택). 후보: "v5"

out = K.run(K.TOPICS[TOPIC], rule=RULE, verbose=False)
cb = out["calibration"]
print(f"보정 기간에서 고른 값: α = {cb.alpha}, m = {cb.m:.3f} · 이탈 해 |이탈률| ≥ {cb.dev:.1%}, 정상 해 < {cb.clear:.1%}, φ = {cb.phi:.2f}")
display(pd.Series(out["test"], name="채점 기간"))
out["groups"]


In [ ]:

# 실시간 경보 목록 (올해 공표분)
out["live"]


In [ ]:

# 리포트 한 장 (파일로도 저장)
md = K.report(out)
open(f"report_{TOPIC}.md", "w", encoding="utf-8").write(md)
display(Markdown(md))



## 새 주제 추가하기

1. **찾기:** `K.search("시군구 월별 출생")` → 표 번호(`table`)와 기관(`org`) 확인
2. **구조 보기:** `K.describe("DT_1B81A01")` → 월별 자료가 있는지(주기), 분류마다 넣을 값(`objL1`, `objL2` …)과 항목(`itmId`)
3. **설정 만들기:** 아래 예시처럼 `Topic`을 정의합니다. 바꿔야 하는 것은 **주제의 사실**뿐입니다.
   - 자료 종류(`kind`): 흐름(건수) / 순흐름(전입−전출) / 재고(수준)
   - 지역 단위, 기간, 평가 대상 최소 규모, 규모 구간
4. **실행:** `K.run(topic)` → 처음 한 번은 KOSIS에서 받고, `cache` 파일로 저장해 다음부터 바로 돕니다.

판정 절차(통일 규칙)와 기준을 정하는 절차(보정 기간, 잘못된 경보 ≤ 10%)는 **바꾸지 않습니다.**


In [ ]:

NEW = False   # True로 바꾸면 실행 (KOSIS_API_KEY 필요)
if NEW:
    births = K.Topic(
        name="시군구 출생", table="DT_1B81A01", kind="flow",
        series={"y": K.Series("T1", {"objL1": "ALL"})},          # T1 = 계 (남+여)
        unit_cols=("C1",), unit_filter=lambda u, n: len(u) == 5 and u.endswith("0") and "변동전" not in n,
        start="199701", end="202512", cal=range(2003, 2013), test=range(2013, 2026), live_year=2025,
        min_size=300, thresholds=("fixed", 0.10, 0.05),
        size_cuts=(0, 1000, 3000, float("inf")), size_labels=("300~1천", "1천~3천", "3천 이상"),
        cache="data_kosis/pipeline_births.csv")
    K.TOPICS["출생"] = births
    out = K.run(births)
    display(Markdown(K.report(out)))
